# Snack detection

### Snackverpackungen mit Kamera erkennen

In [1]:
from pathlib import Path
import hashlib
import random
import shutil
import subprocess
import sys

import yaml
from IPython.display import display, Markdown, Image

import camera

ROOT = Path.cwd()
IMAGE_TYPES = {'jpg', 'jpeg', 'png'}

if not (ROOT / "camera.py").is_file():
    raise RuntimeError(f"File not found: {ROOT / 'camera.py'}")

for folder in ["data/raw", "data/export", "models"]:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)


print("Project set up!")


Project set up!


# Open camera

In [2]:
camera_process = subprocess.Popen(
        [sys.executable, "camera.py", "--preview", "--camera", "0"],
        cwd=ROOT,
    )

# Fotos anzeigen

In [3]:
photos = sorted(
    path
    for path in (ROOT / "data/raw").iterdir()
    if path.suffix.lower() in IMAGE_TYPES
)

display(Markdown(f"### Gesammelte Bilder: **{len(photos)}**"))

if not photos:
    print("Noch keine Bilder vorhanden.")
else:
    for photo in photos[-3:]:
        display(Image(filename=str(photo), width=350))

### Gesammelte Bilder: **0**

Noch keine Bilder vorhanden.


# Label Studio vorbereiten

#### Linux/Mac

```` bash
python3.12 -m venv .venv-labelstudio

.venv-labelstudio/bin/python -m pip install label-studio==1.23.2

.venv-labelstudio/bin/label-studio start --data-dir .label-studio --internal-host 127.0.0.1 --port 8080
````

#### Windows
```bash
py -3.12 -m venv .venv-labelstudio

.\.venv-labelstudio\Scripts\python.exe -m pip install label-studio==1.23.2

.\.venv-labelstudio\Scripts\label-studio.exe start --data-dir .label-studio --internal-host 127.0.0.1 --port 8080
```

# In Label Studio:
1. http://localhost:8080 öffnen.
2. Lokales Konto und Projekt erstellen.
3. Unter Settings → Labeling Interface → Code den XML-Inhalt einfügen.
4. Später die Bilder aus data/raw/ importieren.
5. Jeden sichtbaren Snack mit einer Box beschriften und mit Submit speichern.
6. Als YOLO with Images exportieren.
7. ZIP nach data/export/ entpacken.

# Datensatz vorbereiten

- **80 %** der Bilder zum Trainieren
- **20 %** zur Validierung

In [4]:
def prepare_dataset():
    output = ROOT / "data/dataset"

    if output.exists():
        raise ValueError("dataset exists already.")

    exports = list((ROOT / "data/export").rglob("classes.txt"))

    if len(exports) != 1:
        raise ValueError("Extract YOLO-Export")

    export = exports[0].parent
    names = exports[0].read_text(encoding="utf-8-sig").splitlines()
    names = [name.strip() for name in names]

    if not names or any (not name for name in names):
        raise ValueError("No names in classes.txt")

    if len(names) != len(set(names)):
        raise ValueError("Duplicate names in classes.txt")

    images = sorted(
        path
        for path in (export / "images").rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_TYPES
    )

    records = []
    seen_names = set()
    seen_hashes = set()

    for image in images:
        digest = hashlib.sha256(image.read_bytes()).hexdigest()

        if image.stem in seen_names or digest in seen_hashes:
            raise ValueError(f"Duplicate image: {image.name}")

        seen_names.add(image.stem)
        seen_hashes.add(digest)

        relative = image.relative_to(export / "images")
        label = export / "labels" / relative.with_suffix(".txt")

        if not label.is_file():
            raise ValueError(f"Label not found: {label.name}")

        classes = set()

        for line in label.read_text(encoding="utf-8-sig").splitlines():
            if not line.strip():
                continue

            parts = line.split()

            if len(parts) != 5:
                raise ValueError(f"Invalid label: {line.name}")

            class_id = int(parts[0])
            x, y, width, height = map(float, parts[1:])

            if not (
                0 <= class_id < len(classes)
                and 0 <= x <= 1
                and 0 <= y <= 1
                and 0 < width <= 1
                and 0 < height <= 1
            ):
                raise ValueError(f"Invalid label: {line.name}")

            classes.add(class_id)

        records.append((image, label, classes))

    if len(records) < 2:
        raise ValueError("not enough images")

    expected = set(range(len(names)))
    validation_count = max(1, round(len(records) * 0.2))
    rng = random.Random(42)

    for _ in range(500):
        rng.shuffle(records)

        validation = records[:validation_count]
        training = records[validation_count:]

        train_classes = set().union(*(item[2] for item in training))
        val_classes = set().union(*(item[2] for item in validation))

        if train_classes == expected and val_classes == expected:
            break

        else:
            raise ValueError("You need more images for each class")

        for split, items in [("train", training), ("val", validation)]:
            image_folder = output / "images" / split
            label_folder = output / "labels" / split

            image_folder.mkdir(parents=True)
            label_folder.mkdir(parents=True)

            for image, label, _ in items:
                shutil.copy2(image, image_folder / image.name)
                shutil.copy2(label, label_folder / f"{image.stem}.txt")

        config = {
            "path": str(output.resolve()),
            "train": "images/train",
            "val": "images/val",
            "names": dict(enumerate(names)),
        }

        (output / "data.yaml").write_text(
            yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
            encoding="utf-8",
        )


    display(Markdown(f"""
### Datensatz vorbereitet

| Bereich | Bilder |
|---|---:|
| Training | {len(training)} |
| Validierung | {len(validation)} |
| Gesamt | {len(records)} |

**Klassen:** {", ".join(names)}
"""))


try:
    prepare_dataset()
except ValueError as error:
    print(error)

Extract YOLO-Export


# Check GPU

In [5]:
import torch

print("CUDA verfügbar:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA verfügbar: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


# YOLO trainieren

**YOLO11n** wird verwendet


In [7]:
from ultralytics import YOLO

EPOCHS = 50
run_dir = None

dataset = ROOT / "data/dataset/data.yaml"

if not dataset.is_file():
    print("Dataset not found")

else:
    model = YOLO(str(ROOT / "models/yolo11n.pt"))

    model.train(
        data=str(dataset),
        epochs=EPOCHS,
        batch=4,
        imgsz=640,
        device="gpu",
        workers=0,
        project=str(ROOT / "runs"),
        name="snacks",
        plots=True,
        seed=42,
    )

    best_weights = Path(model.trainer.best)

    if not best_weights.is_file():
        raise RuntimeError("No weights found")

    shutil.copy2(best_weights, ROOT / "models/best.pt")
    run_dir = Path(model.trainer.run_dir)

    display(Markdown("""
        ### Training abgeschlossen
        Beste Modell ist in **`models/best.pt`**
    """))


WARNING ⚠️ torchvision==0.29 is incompatible with torch==2.11.
Run 'pip install torchvision==0.26' to fix torchvision or 'pip install -U torch torchvision' to update both.
For a full compatibility table see https://github.com/pytorch/vision#installation
WARNING ⚠️ Ultralytics settings updated to the latest schema. Existing values were preserved where possible. 
View Ultralytics Settings with 'yolo settings' or at '/home/fekdac/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Dataset not found


# Ergebnisse

Trainingsverlauf zeigt, wie sich Fehler und Erkennungsmetriken entwickeln.

Die Confusion Matrix zeigt, welche Klasen das Modell erkennt oder verwechselt.

In [8]:
if run_dir is None:
    print("run training first")

else:
    for title, filename in [
        ("Trainingsverlauf", "result.png"),
        ("Confusionmatrix", "confusion_matrix_normalized.png"),
    ]:
        path = run_dir / filename

        if path.is_file():
            display(Markdown(f"### {title}"))
            display(Image(filename=str(path), width=900))

run training first


# Live-Erkennung

Laptop-Kamera Test

**Q** oder **ESC** schließt Fenster

In [9]:
CONFIDENCE = 0.4
CAMERA = 0

if not (ROOT / "models/best.pt").is_file():
    print("Bitte zuerst ein Snack-Modell trainieren.")
elif "camera_process" in globals() and camera_process.poll() is None:
    print("Bitte zuerst das vorhandene Kamerafenster mit Q schließen.")
else:
    camera_process = subprocess.Popen(
        [
            sys.executable,
            "camera.py",
            "--camera", str(CAMERA),
            "--conf", str(CONFIDENCE),
        ],
        cwd=ROOT,
    )

    display(Markdown("### Kamerafenster mit Snack-Erkennung gestartet"))

Bitte zuerst ein Snack-Modell trainieren.
